# Wastewater image regression: model comparison
Set the runtime to a GPU first: **Runtime → Change runtime type → T4 GPU**. Then run the cells from top to bottom.

In [ ]:
import tensorflow as tf
print('GPUs:', tf.config.list_physical_devices('GPU'))  # should list one GPU

In [ ]:
import getpass, os
GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <- change this
REPO = "wastewater-training"
if not os.path.exists(REPO):
    token = getpass.getpass("GitHub token (press Enter if the repo is public): ")
    auth = f"{GITHUB_USER}:{token}@" if token else ""
    !git clone https://{auth}github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!git pull

## Quick test (one model)
Check that everything works before the long runs.

In [ ]:
!python train.py MobileNetV2
!cp results.csv results_test.csv

## Full runs (in batches, so a Colab disconnect does not lose everything)

In [ ]:
!python train.py CNN EfficientNetB0 MobileNetV2
!cp results.csv results_batch1.csv

In [ ]:
!python train.py ResNet50 Xception
!cp results.csv results_batch2.csv

In [ ]:
!python train.py VGG16 VGG19
!cp results.csv results_batch3.csv

## Combine and download the results

In [ ]:
import glob, pandas as pd
from google.colab import files
r = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("results_batch*.csv"))])
r.to_csv("results_all.csv", index=False)
print(r.pivot(index="model", columns="target", values="R2").round(3).to_string())
files.download("results_all.csv")

## Report for the teacher
Builds R² tables, confusion matrices (Low / Medium / High) and scatter plots from the saved predictions. Run this after all three batches.

In [ ]:
!python make_report.py
!zip -qr report.zip report
from google.colab import files
files.download("report.zip")

## Actual vs predicted in original units (mg/L, NTU, pH)

In [ ]:
!python make_comparison.py
!zip -qr report.zip report
from google.colab import files
files.download("report.zip")

## Per-photo report (measured values + every model's prediction)

In [ ]:
!python make_image_report.py
!zip -qr report.zip report
from google.colab import files
files.download("report.zip")